In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import factorial
from finite_diff_functions import *

# Exercise 2a
Solving the equation
$$
\begin{cases}

    \epsilon u'' + u'(u - 1) = f, & \quad 0 \le t \le 1 \\
    u(0) = \alpha, \\ u(1)= \beta
\end{cases}
$$
with DC boundary conditions
## (a)

## (b)

First we code up for a global linear BVP:

In [ ]:
eps = 0.1
a = 0
b = 1
alpha = 0
beta = np.sin(1)
w0 = 0.5*(a - b + beta - alpha)
xtilde = 0.5*(a + b - alpha - beta)

# mesh
N = 4
x = np.linspace(a, b, N)
h = x[1] - x[0] # This grid is uniform

# Method of Manufactured solution (MoMS)
u_moms = lambda x: np.sin(x)
f = - (eps + 1) * np.sin(x)

stencil = np.array([-1, 0, 1])

D2 = fdcoeffV_uniform(2, x[1] - x[0], stencil) * eps
D0 = fdcoeffV_uniform(1, x[1] - x[0], stencil)
# D2 = fdcoeffV(2, x[1] - x[0], stencil) * eps
# D0 = fdcoeffV(1, x[1] - x[0], stencil)
# D2, D0
sub_diag = np.full(N-1, D2[0]) # 
d = np.full(N, D2[1])
sup_diag = np.full(N-1, D2[2])

A = np.diag(sub_diag, -1) + np.diag(d) + np.diag(sup_diag, 1) - np.eye(N)
b = np.copy(f)

# IMPOSE BC
## For RHS
b[0] = alpha
b[-1] = beta
b[1] -= A[1, 0]*alpha
b[-2] -= A[-2, -1]*beta

## System matrix
A[0,0] = 1
A[1, 0] = 0
A[0, 1] = 0

A[-1,-1] = 1
A[-2, -1] = 0
A[-1, -2] = 0

In [ ]:
u_FD = np.linalg.solve(A, b)

In [ ]:
plt.plot(np.linspace(0,1, 100), u_moms(np.linspace(0,1,100)), '--', label = 'anal')
plt.plot(x, u_FD, 'ro-', label = 'FD', alpha = 0.75)
plt.legend()
plt.show()

G function for later

# Convergence test

In [ ]:
grid_points = np.array([4, 10, 25, 50], dtype=np.int16)
h = (b - a) / (grid_points - 1)
errs = []

for N in grid_points:
    # mesh
    x = np.linspace(a, b, N)

    # Method of Manufactured solution (MoMS)
    u_moms = lambda x: np.sin(x)
    f = - (eps + 1) * np.sin(x)

    stencil = np.array([-1, 0, 1])

    D2 = fdcoeffV_uniform(2, x[1] - x[0], stencil) * eps
    D0 = fdcoeffV_uniform(1, x[1] - x[0], stencil)
    # D2 = fdcoeffV(2, x[1] - x[0], stencil) * eps
    # D0 = fdcoeffV(1, x[1] - x[0], stencil)
    # D2, D0
    sub_diag = np.full(N-1, D2[0])
    d = np.full(N, D2[1])
    sup_diag = np.full(N-1, D2[2])

    A = np.diag(sub_diag, -1) + np.diag(d) + np.diag(sup_diag, 1) - np.eye(N)

    u_FD = np.linalg.solve(A, f)

    errs.append(l2_norm(u_moms(x) - u_FD))

In [ ]:
plt.rc('text', usetex=True)
plt.rc('font', family='serif')


plt.loglog(h,15*h**3,"b--", label=r"$\mathcal O(h^3)$")
plt.loglog(h,errs,color="orange", label=r"$(\alpha, \beta) = (2,2)$")
plt.title("Convergence for Newtons method")
plt.grid(True, which = 'both')
plt.legend()
plt.show()

### Non-linear

In [ ]:
eps = 0.1
a = 0
b = 1
alpha = -1
beta = 1.5
w0 = 0.5*(a - b + beta - alpha)
xtilde = 0.5*(a + b - alpha - beta)

# mesh
N = 4
x = np.linspace(a, b, N)
h = x[1] - x[0] # This grid is uniform

initial_guess = x - xtilde + w0*np.tanh(w0*(x - xtilde) / 2 * eps)


# Method of Manufactured solution (MoMS)
u_moms = lambda x: np.sin(x)
f = (np.cos(x) - eps - 1) * np.sin(x)

stencil = np.array([-1, 0, 1])

D2 = fdcoeffV_uniform(2, x[1] - x[0], stencil) * eps
D0 = fdcoeffV_uniform(1, x[1] - x[0], stencil)


sub_diag = np.full(N-1, D2[0]) # 
d = np.full(N, D2[1])
sup_diag = np.full(N-1, D2[2])

A = np.diag(sub_diag, -1) + np.diag(d) + np.diag(sup_diag, 1) - np.eye(N)
b = np.copy(f)

In [ ]:
uk = initial_guess
uk

In [ ]:
A

In [ ]:
def G(uk, A = A): # G(u) = A @ u + non_linear(u) - F
    eval_g = np.zeros_like(uk)
    eval_g[1:-1] = (A @ uk)[1:-1] - f[1:-1] + uk[1:-1] * ((uk[2:] - uk[:-2]) / (2*h))
    return eval_g

def Jacobian(uk, A = A, h = h):
    sub_diag = np.diag(uk[1:], -1)
    sup_diag = np.diag(uk[:-1], -1)
    main = np.eye(uk.size)
    main[1:-1, 1:-1] = np.diag(uk[2:] - uk[:-2])
    return A + 1/(2*h) * (-sub_diag + main + sup_diag)

In [ ]:
Jacobian(uk)

In [ ]:
G(uk)